# Comparar modelos — Directo vs CoT (Colab opcional · e7)

Este notebook es el **camino opcional** del ejercicio complementario e7. Pasa **un mismo problema de
razonamiento** por **2–3 modelos** (uno por proveedor) en **dos condiciones** — *directo* vs
*Chain-of-Thought ("pensemos paso a paso")* — y arma una **tabla lado a lado** para que puntúes con
la rúbrica de 4 ejes.

**Está pensado para no programadores.** Solo tienes que:
1. Ejecutar las celdas **de arriba hacia abajo** (botón ▶ a la izquierda de cada celda, o
   `Shift+Enter`).
2. Pegar tus **API keys** en la celda indicada.
3. Leer la tabla final y puntuar con la rúbrica.

> No necesitas entender el código. Los comentarios en español explican qué hace cada parte.
> Si un modelo falla (sin crédito, sin key), el notebook **no se rompe**: muestra el error en su
> celda y sigue con los demás.

## Paso 1 — Instalar los SDKs

Instala las librerías oficiales de los tres proveedores. En Colab esto tarda ~30 segundos.
Si ves un aviso de "reiniciar entorno", puedes ignorarlo.


In [ ]:
# Instala los SDKs de los tres proveedores (silencioso con -q).
!pip install -q openai anthropic google-genai
print("SDKs instalados. Continua con el Paso 2.")


## Paso 2 — Pega aquí tus API keys

Cada proveedor te da una clave (key) en su panel. **Pégala entre las comillas.** Si no tienes la de
algún proveedor, **déjala vacía**: ese modelo simplemente se salta y los demás siguen funcionando.

Dónde obtener cada key:
- **OpenAI** → https://platform.openai.com/api-keys
- **Anthropic** → https://console.anthropic.com/settings/keys
- **Google (Gemini)** → https://aistudio.google.com/apikey

> ⚠️ **No compartas estas claves ni subas el notebook con las claves pegadas.** Son como una
> contraseña: dan acceso a tu cuenta y pueden generar gastos. Si la expones por error, bórrala
> (revoke) en el panel del proveedor y crea una nueva.


In [ ]:
# Pega cada clave entre las comillas. Deja "" (vacio) la que no tengas.
OPENAI_API_KEY    = ""   # ej: "sk-..."
ANTHROPIC_API_KEY = ""   # ej: "sk-ant-..."
GOOGLE_API_KEY    = ""   # ej: "AIza..."

# Guarda las claves como variables de entorno (forma estandar de usarlas).
import os
os.environ["OPENAI_API_KEY"]    = OPENAI_API_KEY
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["GOOGLE_API_KEY"]    = GOOGLE_API_KEY

print("Claves cargadas (las vacias se ignoraran automaticamente).")


## Paso 3 — El problema y las dos condiciones

Definimos **un solo problema** (el carrito de e1) y **dos versiones del prompt**: directo y CoT.
Es exactamente el mismo contenido de `problema.md`, `prompt-directo.md` y `prompt-cot.md`.

La respuesta correcta es **S/ 1,084.52** (el error comun es S/ 1,056.20). La usamos solo para
marcar la corrección en la tabla.


In [ ]:
# El problema de razonamiento (identico a problema.md).
PROBLEMA = """Calcula el total a pagar de este carrito. Reglas y orden obligatorio: primero los
descuentos, luego el IGV, y al final el cupon.
- Camara: precio S/ 1,200. Descuento del 20% y, sobre el resultado, un 10% adicional (en cascada).
- Libro: precio S/ 90. Exento de IGV, sin descuentos.
- Envio: S/ 25, sin IGV y sin descuentos.
- El IGV es 18% y se aplica solo sobre la camara.
- Cupon de S/ 50 que se resta del total, al final de todo.
Cual es el total a pagar en soles?"""

# Condicion A: prompt directo (sin razonamiento).
PROMPT_DIRECTO = (
    "Resuelve el siguiente problema y responde SOLO con el total final en soles, "
    "sin explicacion ni pasos.\n\n" + PROBLEMA
)

# Condicion B: prompt CoT ("pensemos paso a paso").
PROMPT_COT = (
    "Resuelve el siguiente problema. Pensemos paso a paso: muestra tu razonamiento numerado, "
    "una operacion por paso, y al final escribe la respuesta en una linea separada con el "
    'formato exacto "Respuesta final: S/ ___".\n\n' + PROBLEMA
)

RESPUESTA_CORRECTA = "1084.52"   # error comun: 1056.20

# Temperatura: 0.0 = respuesta estable; subela (ej. 0.7) para ver variabilidad / self-consistency.
TEMPERATURA = 0.0

print("Problema y condiciones definidos.")


## Paso 4 — Funciones que llaman a cada modelo

Una función por proveedor. Cada una recibe el texto del prompt y devuelve la respuesta del modelo.
Todas usan `try/except`: si algo falla (sin key, sin crédito, error de red), devuelven el mensaje de
error como texto en vez de romper el notebook.

> **Nombres de modelos:** los proveedores cambian sus modelos seguido. Si un nombre quedara
> obsoleto, reemplaza el valor de la variable `MODELO_*` por el modelo disponible que veas en el
> panel del proveedor (AI Studio / Console / Playground).


In [ ]:
# Ajusta estos nombres si el proveedor renombro sus modelos (ver panel de cada uno).
MODELO_OPENAI    = "gpt-4o-mini"
MODELO_ANTHROPIC = "claude-3-5-haiku-latest"
MODELO_GOOGLE    = "gemini-1.5-flash"


def responder_openai(prompt, temperatura=TEMPERATURA):
    """Llama a un modelo de OpenAI. Devuelve el texto o un mensaje de error."""
    try:
        from openai import OpenAI
        if not os.environ.get("OPENAI_API_KEY"):
            return "[OMITIDO] No hay OPENAI_API_KEY."
        client = OpenAI()
        resp = client.chat.completions.create(
            model=MODELO_OPENAI,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperatura,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"[ERROR OpenAI] {e}"


def responder_anthropic(prompt, temperatura=TEMPERATURA):
    """Llama a un modelo de Anthropic (Claude). Devuelve el texto o un mensaje de error."""
    try:
        import anthropic
        if not os.environ.get("ANTHROPIC_API_KEY"):
            return "[OMITIDO] No hay ANTHROPIC_API_KEY."
        client = anthropic.Anthropic()
        resp = client.messages.create(
            model=MODELO_ANTHROPIC,
            max_tokens=1024,
            temperature=temperatura,
            messages=[{"role": "user", "content": prompt}],
        )
        # La respuesta viene como una lista de bloques; tomamos el texto.
        return "".join(b.text for b in resp.content if getattr(b, "type", "") == "text").strip()
    except Exception as e:
        return f"[ERROR Anthropic] {e}"


def responder_google(prompt, temperatura=TEMPERATURA):
    """Llama a un modelo de Google (Gemini). Devuelve el texto o un mensaje de error."""
    try:
        from google import genai
        from google.genai import types
        if not os.environ.get("GOOGLE_API_KEY"):
            return "[OMITIDO] No hay GOOGLE_API_KEY."
        client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
        resp = client.models.generate_content(
            model=MODELO_GOOGLE,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=temperatura),
        )
        return (resp.text or "").strip()
    except Exception as e:
        return f"[ERROR Google] {e}"


print("Funciones listas.")


## Paso 5 — Correr las dos condiciones en todos los modelos

Recorremos cada modelo en cada condición (directo y CoT) y guardamos las respuestas. Esto hace las
llamadas reales a las APIs; puede tardar unos segundos por modelo.


In [ ]:
# Mapa de proveedores -> funcion. Comenta una linea si no quieres usar ese proveedor.
MODELOS = {
    "OpenAI ("    + MODELO_OPENAI    + ")": responder_openai,
    "Anthropic (" + MODELO_ANTHROPIC + ")": responder_anthropic,
    "Google ("    + MODELO_GOOGLE    + ")": responder_google,
}

CONDICIONES = {
    "Directo": PROMPT_DIRECTO,
    "CoT":     PROMPT_COT,
}

resultados = []  # cada item: (modelo, condicion, respuesta)
for nombre_modelo, fn in MODELOS.items():
    for nombre_cond, prompt in CONDICIONES.items():
        print(f"Llamando: {nombre_modelo} | {nombre_cond} ...")
        salida = fn(prompt)
        resultados.append((nombre_modelo, nombre_cond, salida))

print("\nListo. Respuestas recolectadas:", len(resultados))


## Paso 6 — Tabla lado a lado

Mostramos todas las respuestas en una tabla con pandas. La columna **¿Acierta?** marca con un check
las salidas cuyo texto contiene la respuesta correcta (`1084.52`). Es una ayuda automatica; la
**corrección final la confirmas tú** leyendo la respuesta.


In [ ]:
import pandas as pd

def marca_correccion(texto):
    """Marca de ayuda: busca la respuesta correcta dentro del texto (ignora la coma de miles)."""
    limpio = texto.replace(",", "")
    if RESPUESTA_CORRECTA in limpio:
        return "OK (1084.52)"
    if "1056.20" in limpio or "1056.2" in limpio:
        return "Error comun (1056.20)"
    return "revisar a mano"

filas = []
for modelo, condicion, salida in resultados:
    filas.append({
        "Modelo": modelo,
        "Condicion": condicion,
        "Respuesta del modelo": salida,
        "¿Acierta? (auto)": marca_correccion(salida),
    })

df = pd.DataFrame(filas)
pd.set_option("display.max_colwidth", 120)  # que no corte el texto
df


## Paso 7 — Puntúa con la rúbrica (lo haces tú)

La tabla de arriba te da las salidas; ahora **puntúalas** con la rúbrica de 4 ejes de
`rubrica-comparacion.md` (escala 1–5; eje 1 eliminatorio; eje 2 = N/A en la condición directa):

1. **Corrección** — ¿el total es S/ 1,084.52? (binario/eliminatorio)
2. **Fidelidad de los pasos** — ¿cada paso es válido y lleva a la respuesta? (N/A si es directo)
3. **Claridad** — ¿el rastro es legible y auditable?
4. **Coste** — longitud/tokens/latencia (más pasos = más caro).

Edita el diccionario de abajo con tus puntajes y ejecuta para ver tu tabla de evaluación y un
ganador sugerido **por esta tarea** (mayor corrección, desempate por suma de los otros ejes).


In [ ]:
# Rellena tus puntajes (1-5). Usa None para "N/A" (p. ej. fidelidad en la condicion directa).
# Las claves deben coincidir con (Modelo, Condicion) tal como aparecen en la tabla del Paso 6.
puntajes = {
    # ("OpenAI (gpt-4o-mini)", "Directo"): {"correccion": 1, "fidelidad": None, "claridad": 5, "coste": 5},
    # ("OpenAI (gpt-4o-mini)", "CoT"):     {"correccion": 5, "fidelidad": 5,    "claridad": 5, "coste": 3},
    # ... agrega una linea por cada celda que evaluaste ...
}

if not puntajes:
    print("Aun no cargaste puntajes. Descomenta y completa el diccionario 'puntajes' y vuelve a ejecutar.")
else:
    eval_filas = []
    for (modelo, condicion), p in puntajes.items():
        otros = [v for k, v in p.items() if k != "correccion" and isinstance(v, (int, float))]
        eval_filas.append({
            "Modelo": modelo,
            "Condicion": condicion,
            "Correccion": p.get("correccion"),
            "Fidelidad": p.get("fidelidad"),
            "Claridad": p.get("claridad"),
            "Coste": p.get("coste"),
            "Suma otros ejes": sum(otros),
        })
    eval_df = pd.DataFrame(eval_filas)
    # Ganador: primero correccion mas alta, luego mayor suma de los demas ejes.
    eval_df = eval_df.sort_values(["Correccion", "Suma otros ejes"], ascending=False)
    ganador = eval_df.iloc[0]
    print(f"Ganador sugerido POR ESTA TAREA: {ganador['Modelo']} | {ganador['Condicion']}")
    print("(Confirma siempre con criterio: corre+coste, no tamano del modelo.)\n")
    display(eval_df)


## (Opcional) Self-consistency con este notebook

Para reproducir **e4** aquí: sube `TEMPERATURA` a `0.7` en el Paso 3, vuelve a ejecutar el Paso 5
varias veces (5 corridas) anotando la "Respuesta final" de cada una, y aplica **voto mayoritario**.
Las cadenas correctas tienden a converger a S/ 1,084.52; los errores se dispersan.